# 96 — G-N: kéo danh sách bài trích dẫn và kết quả tìm kiếm qua API

Mục đích: lấy **dữ liệu thô** cho cổng G-N để Claude tự lọc và điền bảng. Người chạy không phải đọc hay đánh giá bài nào.

Nguồn (đều miễn phí, không cần tài khoản):
- **Semantic Scholar Graph API**: danh sách bài trích dẫn (citations) và tìm theo từ khoá.
- **OpenAlex API**: danh sách bài trích dẫn (filter `cites:`) và tìm theo từ khoá. Dùng để đối chiếu với Semantic Scholar.
- **arXiv API**: tìm toàn văn tóm tắt theo truy vấn có trường.

Google Scholar không có API và chặn máy tự truy cập, nên không dùng ở đây.

Cách chạy: CPU, **bật Internet**, chạy hết (khoảng 5–10 phút, chủ yếu là chờ giới hạn tốc độ của API).
Trên Kaggle: chạy từng ô (QT-11, trường hợp d) rồi **Stop session**; hoặc chạy trên máy cá nhân.
Kết quả: `OUT_DIR.zip`. Mang file zip này vào phiên chat.

Mọi phản hồi gốc của API được lưu nguyên trong `raw/`. Nếu phần phân tích của notebook sai, Claude vẫn đọc lại được từ dữ liệu gốc.

In [ ]:
# Tham số (papermill). Comment không có dấu bằng (QT-21).
OUT_DIR = "outputs/96_gn_tra_cuu"
# Khoá Semantic Scholar nếu có (không bắt buộc); để trống thì dùng hạn mức chung
S2_API_KEY = ""
# Email cho hàng ưu tiên của OpenAlex (không bắt buộc); để trống cũng chạy
OPENALEX_MAILTO = ""
YEAR_FROM = 2024
MAX_PER_QUERY = 100
# Chế độ thử không mạng: dùng phản hồi giả để kiểm phần phân tích
TOY_TEST = False

## Ô 1. Danh sách bài gốc và truy vấn

Mỗi bài gốc có nhiều mã (arXiv, DOI của ACL) vì OpenAlex đôi khi tách bản arXiv và bản hội nghị thành hai mục, làm trích dẫn bị chia đôi.
Mã nào không tìm thấy thì ghi vào `errors.jsonl`, không dừng notebook.

In [ ]:
import os, sys, json, time, gzip, re, hashlib, platform, shutil
import xml.etree.ElementTree as ET
print("python", platform.python_version())

SEEDS = [
    # nhom chinh: 7 bai trong GN_TRA_CUU.md
    {"key": "causalgym", "nhom": "chinh", "title": "CausalGym: Benchmarking causal interpretability methods on linguistic tasks",
     "s2": ["arXiv:2402.12560", "DOI:10.18653/v1/2024.acl-long.785"], "doi": ["10.48550/arXiv.2402.12560", "10.18653/v1/2024.acl-long.785"]},
    {"key": "ouyang", "nhom": "chinh", "title": "Low-Bit Quantization Favors Undertrained LLMs",
     "s2": ["arXiv:2411.17691", "DOI:10.18653/v1/2025.acl-long.1555"], "doi": ["10.48550/arXiv.2411.17691", "10.18653/v1/2025.acl-long.1555"]},
    {"key": "duan", "nhom": "chinh", "title": "Perplexity Can Miss SAE Feature Damage Under Quantization",
     "s2": ["arXiv:2606.03002"], "doi": ["10.48550/arXiv.2606.03002"]},
    {"key": "makarov", "nhom": "chinh", "title": "One Vector or Many? Clustered Optimal-Transport Steering under Quantization",
     "s2": ["arXiv:2610.40055"], "doi": ["10.48550/arXiv.2610.40055"]},
    {"key": "tanyel", "nhom": "chinh", "title": "Whose Effect Is It? How Objectives Inflate Learned Subspace Interventions",
     "s2": ["arXiv:2611.05033"], "doi": ["10.48550/arXiv.2611.05033"]},
    {"key": "geiger_das", "nhom": "chinh", "title": "Finding Alignments Between Interpretable Causal Variables and Distributed Neural Representations",
     "s2": ["arXiv:2303.02536"], "doi": ["10.48550/arXiv.2303.02536"]},
    {"key": "zhou", "nhom": "chinh", "title": "From Signal Degradation to Computation Collapse: Uncovering the Two Failure Modes of LLM Quantization",
     "s2": ["arXiv:2604.19884", "DOI:10.18653/v1/2026.findings-acl.1162"], "doi": ["10.48550/arXiv.2604.19884", "10.18653/v1/2026.findings-acl.1162"]},
    # nhom phu: bai lan can gan C1 va C3b
    {"key": "bhandari", "nhom": "phu", "title": "Steering Under Compression", "s2": ["arXiv:2609.06473"], "doi": ["10.48550/arXiv.2609.06473"]},
    {"key": "wu_margin", "nhom": "phu", "title": "Which Decisions Low-Bit Quantization Breaks, and How to Predict Them", "s2": ["arXiv:2608.06564"], "doi": ["10.48550/arXiv.2608.06564"]},
    {"key": "yao_circuit", "nhom": "phu", "title": "The Emergence and Sparsification of a Syntactic Circuit", "s2": ["arXiv:2610.33299"], "doi": ["10.48550/arXiv.2610.33299"]},
    {"key": "armstrong", "nhom": "phu", "title": "Pythia-160m INT4 divergence across checkpoints", "s2": ["arXiv:2604.15167"], "doi": ["10.48550/arXiv.2604.15167"]},
]

KEYWORD_QUERIES = [
    "quantization causal abstraction",
    "quantized distributed alignment search",
    "interchange intervention quantized language model",
    "CausalGym quantization",
    "mechanistic interpretability quantization Pythia",
    "activation patching quantized model training checkpoints",
    "DAS GPTQ interpretability",
    "subject-verb agreement quantization mechanism",
    "interpretability findings transfer to quantized models",
    "quantization-induced degradation mechanism training checkpoints",
]

ARXIV_QUERIES = [
    'all:"causal abstraction" AND all:quantization',
    'all:"distributed alignment search" AND (all:quantization OR all:quantized)',
    'all:"interchange intervention" AND (all:quantization OR all:quantized)',
    'all:CausalGym',
    'all:"activation patching" AND (all:quantization OR all:quantized)',
    'all:"mechanistic interpretability" AND (all:quantization OR all:quantized)',
    'abs:quantization AND abs:interpretability AND abs:checkpoints',
]
print(len(SEEDS), "bài gốc |", len(KEYWORD_QUERIES), "truy vấn từ khoá |", len(ARXIV_QUERIES), "truy vấn arXiv")

## Ô 2. Hàm gọi API (có lưu phản hồi gốc, thử lại khi bị giới hạn tốc độ)

Semantic Scholar không khoá trả lỗi 429 khá thường; hàm chờ tăng dần (5, 10, 20, 40, 80 giây) rồi thử lại.
arXiv yêu cầu cách nhau ít nhất 3 giây.

In [ ]:
os.makedirs(os.path.join(OUT_DIR, "raw"), exist_ok=True)
F_ERR = os.path.join(OUT_DIR, "errors.jsonl"); open(F_ERR, "w").close()
def ghi_loi(rec):
    with open(F_ERR, "a") as f: f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def luu_raw(kind, url, params, text):
    h = hashlib.sha1((url + json.dumps(params, sort_keys=True)).encode()).hexdigest()[:12]
    with gzip.open(os.path.join(OUT_DIR, "raw", f"{kind}_{h}.json.gz"), "wt") as f:
        json.dump({"url": url, "params": params, "body": text}, f)

if not TOY_TEST:
    import requests
    SESSION = requests.Session()
    SESSION.headers["User-Agent"] = "khoaluan-gn-check/1.0"

_last = {}
def http_get(kind, url, params=None, headers=None, min_gap=1.2):
    params = params or {}
    if TOY_TEST:
        return toy_response(kind, url, params)
    gap = 3.1 if kind.startswith("arxiv") else min_gap
    host = url.split("/")[2]
    wait = gap - (time.time() - _last.get(host, 0))
    if wait > 0: time.sleep(wait)
    for attempt in range(6):
        try:
            r = SESSION.get(url, params=params, headers=headers or {}, timeout=60)
            _last[host] = time.time()
        except Exception as e:
            ghi_loi({"kind": kind, "url": url, "params": params, "loi": repr(e)}); time.sleep(5 * 2 ** attempt); continue
        if r.status_code == 429 or r.status_code >= 500:
            time.sleep(5 * 2 ** attempt); continue
        luu_raw(kind, url, params, r.text)
        if r.status_code != 200:
            ghi_loi({"kind": kind, "url": url, "params": params, "status": r.status_code, "body": r.text[:300]})
            return None
        return r.text
    ghi_loi({"kind": kind, "url": url, "params": params, "loi": "hết lượt thử"})
    return None

S2 = "https://api.semanticscholar.org/graph/v1"
S2_H = {"x-api-key": S2_API_KEY} if S2_API_KEY else {}
S2_FIELDS = "title,abstract,year,venue,externalIds,url,authors,publicationDate"
OA = "https://api.openalex.org"
OA_SELECT = "id,doi,display_name,publication_year,abstract_inverted_index,primary_location,ids,authorships"
def oa_params(p):
    if OPENALEX_MAILTO: p = {**p, "mailto": OPENALEX_MAILTO}
    return p

def oa_abstract(inv):
    if not inv: return None
    pos = []
    for w, idx in inv.items():
        for i in idx: pos.append((i, w))
    return " ".join(w for _, w in sorted(pos))

## Ô 3. Phản hồi giả cho TOY_TEST

Chỉ dùng khi `TOY_TEST = True`. Dạng dữ liệu theo tài liệu công khai của từng API; nếu dạng thật khác, dữ liệu gốc trong `raw/` vẫn đủ để Claude phân tích lại.

In [ ]:
def toy_response(kind, url, params):
    if kind == "s2_paper":
        return json.dumps({"paperId": "P_" + url.rsplit("/", 1)[-1], "title": "Toy seed", "year": 2024, "citationCount": 2,
                           "externalIds": {"ArXiv": "2402.12560"}})
    if kind == "s2_citations":
        return json.dumps({"offset": 0, "data": [
            {"citingPaper": {"paperId": "c1", "title": "Does DAS survive GPTQ? Quantized causal abstraction on CausalGym", "abstract": "We test distributed alignment search under 4-bit quantization of Pythia.",
                             "year": 2026, "venue": "arXiv", "externalIds": {"ArXiv": "2611.00001"}, "url": "u1", "authors": [{"name": "A"}]}},
            {"citingPaper": {"paperId": "c2", "title": "A survey of probing", "abstract": None, "year": 2025, "venue": "ACL",
                             "externalIds": {"DOI": "10.1/x"}, "url": "u2", "authors": [{"name": "B"}]}}]})
    if kind == "s2_search":
        return json.dumps({"total": 1, "offset": 0, "data": [{"paperId": "c1", "title": "Does DAS survive GPTQ? Quantized causal abstraction on CausalGym",
                           "abstract": "x", "year": 2026, "venue": "arXiv", "externalIds": {"ArXiv": "2611.00001"}, "url": "u1", "authors": []}]})
    if kind == "oa_work":
        return json.dumps({"id": "https://openalex.org/W1", "display_name": "Toy seed", "cited_by_count": 1, "publication_year": 2024})
    if kind in ("oa_cites", "oa_search"):
        return json.dumps({"meta": {"count": 1, "next_cursor": None}, "results": [
            {"id": "https://openalex.org/W9", "doi": "https://doi.org/10.48550/arxiv.2611.00001", "display_name": "Does DAS survive GPTQ? Quantized causal abstraction on CausalGym",
             "publication_year": 2026, "abstract_inverted_index": {"We": [0], "test": [1], "DAS": [2]},
             "primary_location": {"source": {"display_name": "arXiv"}}, "ids": {"doi": "x"}, "authorships": [{"author": {"display_name": "A"}}]}]})
    if kind == "arxiv":
        return ('<feed xmlns="http://www.w3.org/2005/Atom"><entry><id>http://arxiv.org/abs/2611.00001v1</id>'
                '<title>Does DAS survive GPTQ? Quantized causal abstraction on CausalGym</title><summary>We test DAS.</summary>'
                '<published>2026-11-01T00:00:00Z</published><author><name>A</name></author></entry></feed>')
    return None

## Ô 4. Semantic Scholar: tra bài gốc, rồi lấy toàn bộ bài trích dẫn

In [ ]:
F_CAND = os.path.join(OUT_DIR, "candidates.jsonl"); open(F_CAND, "w").close()
F_SEED = os.path.join(OUT_DIR, "seeds.jsonl"); open(F_SEED, "w").close()
def ghi(path, rec):
    with open(path, "a") as f: f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def s2_rec(p, nguon, goc):
    ext = p.get("externalIds") or {}
    return {"nguon": nguon, "goc": goc, "title": p.get("title"), "year": p.get("year"), "venue": p.get("venue"),
            "abstract": p.get("abstract"), "arxiv": ext.get("ArXiv"), "doi": ext.get("DOI"), "url": p.get("url"),
            "authors": [a.get("name") for a in (p.get("authors") or [])][:6], "date": p.get("publicationDate")}

for s in SEEDS:
    seen_ids = set()
    for pid in s["s2"]:
        t = http_get("s2_paper", f"{S2}/paper/{pid}", {"fields": "paperId,title,year,citationCount,externalIds,venue"}, S2_H)
        if not t: continue
        info = json.loads(t)
        if info.get("paperId") in seen_ids: continue
        seen_ids.add(info.get("paperId"))
        n_got, offset = 0, 0
        while offset is not None:
            t2 = http_get("s2_citations", f"{S2}/paper/{info['paperId']}/citations", {"fields": S2_FIELDS, "limit": 1000, "offset": offset}, S2_H)
            if not t2: break
            page = json.loads(t2)
            for d in page.get("data", []):
                cp = d.get("citingPaper") or {}
                if cp.get("title"):
                    ghi(F_CAND, s2_rec(cp, "s2_citations", s["key"])); n_got += 1
            offset = page.get("next")
        rec = {"nguon": "s2", "goc": s["key"], "nhom": s["nhom"], "ma": pid, "paperId": info.get("paperId"), "title": info.get("title"),
               "citationCount": info.get("citationCount"), "lay_duoc": n_got}
        print(rec); ghi(F_SEED, rec)

## Ô 5. OpenAlex: tra bài gốc theo DOI (bản arXiv và bản hội nghị), lấy toàn bộ bài trích dẫn

In [ ]:
def oa_rec(w, nguon, goc):
    loc = (w.get("primary_location") or {}).get("source") or {}
    return {"nguon": nguon, "goc": goc, "title": w.get("display_name"), "year": w.get("publication_year"), "venue": loc.get("display_name"),
            "abstract": oa_abstract(w.get("abstract_inverted_index")), "arxiv": None, "doi": w.get("doi"), "url": w.get("id"),
            "authors": [(a.get("author") or {}).get("display_name") for a in (w.get("authorships") or [])][:6], "date": None}

for s in SEEDS:
    seen = set()
    for doi in s["doi"]:
        t = http_get("oa_work", f"{OA}/works/https://doi.org/{doi}", oa_params({}))
        if not t: continue
        w = json.loads(t); wid = (w.get("id") or "").rsplit("/", 1)[-1]
        if not wid or wid in seen: continue
        seen.add(wid)
        n_got, cursor = 0, "*"
        while cursor:
            t2 = http_get("oa_cites", f"{OA}/works", oa_params({"filter": f"cites:{wid}", "per-page": 200, "cursor": cursor, "select": OA_SELECT}))
            if not t2: break
            page = json.loads(t2)
            for r in page.get("results", []):
                ghi(F_CAND, oa_rec(r, "oa_citations", s["key"])); n_got += 1
            cursor = (page.get("meta") or {}).get("next_cursor")
            if not page.get("results"): break
        rec = {"nguon": "openalex", "goc": s["key"], "nhom": s["nhom"], "ma": doi, "openalex_id": wid, "title": w.get("display_name"),
               "citationCount": w.get("cited_by_count"), "lay_duoc": n_got}
        print(rec); ghi(F_SEED, rec)

## Ô 6. Tìm theo từ khoá: Semantic Scholar, OpenAlex (từ `YEAR_FROM`), arXiv

In [ ]:
for q in KEYWORD_QUERIES:
    n = 0
    t = http_get("s2_search", f"{S2}/paper/search", {"query": q, "fields": S2_FIELDS, "limit": min(100, MAX_PER_QUERY), "year": f"{YEAR_FROM}-"}, S2_H)
    if t:
        for p in json.loads(t).get("data", []) or []:
            ghi(F_CAND, s2_rec(p, "s2_search", q)); n += 1
    t = http_get("oa_search", f"{OA}/works", oa_params({"search": q, "filter": f"from_publication_date:{YEAR_FROM}-01-01",
                                                         "per-page": min(100, MAX_PER_QUERY), "select": OA_SELECT}))
    if t:
        for r in json.loads(t).get("results", []) or []:
            ghi(F_CAND, oa_rec(r, "oa_search", q)); n += 1
    print(f"{n:4d}  {q}")

NS = {"a": "http://www.w3.org/2005/Atom"}
for q in ARXIV_QUERIES:
    t = http_get("arxiv", "https://export.arxiv.org/api/query", {"search_query": q, "start": 0, "max_results": MAX_PER_QUERY, "sortBy": "relevance"})
    n = 0
    if t:
        try:
            root = ET.fromstring(t)
            for e in root.findall("a:entry", NS):
                aid = (e.findtext("a:id", default="", namespaces=NS) or "").rsplit("/abs/", 1)[-1]
                pub = e.findtext("a:published", default="", namespaces=NS)
                rec = {"nguon": "arxiv_search", "goc": q, "title": " ".join((e.findtext("a:title", default="", namespaces=NS) or "").split()),
                       "year": int(pub[:4]) if pub[:4].isdigit() else None, "venue": "arXiv",
                       "abstract": " ".join((e.findtext("a:summary", default="", namespaces=NS) or "").split()),
                       "arxiv": re.sub(r"v\d+$", "", aid), "doi": None, "url": f"https://arxiv.org/abs/{aid}",
                       "authors": [a.findtext("a:name", default="", namespaces=NS) for a in e.findall("a:author", NS)][:6], "date": pub[:10]}
                ghi(F_CAND, rec); n += 1
        except ET.ParseError as ex:
            ghi_loi({"kind": "arxiv_parse", "query": q, "loi": repr(ex)})
    print(f"{n:4d}  arXiv: {q}")

## Ô 7. Gộp trùng và gắn cờ (chỉ để xếp thứ tự đọc; Claude vẫn đọc tất cả)

Gộp theo tiêu đề đã chuẩn hoá. Cờ: có nhắc lượng tử hoá/nén; có nhắc phương pháp can thiệp (DAS, interchange, patching, steering, SAE, probe); có nhắc CausalGym; có nhắc Pythia hoặc checkpoint.

In [ ]:
def chuan(t): return re.sub(r"[^a-z0-9]+", " ", (t or "").lower()).strip()
RX = {
    "co_luong_tu_hoa": r"quantiz|gptq|awq|\bint ?[248]\b|[248]-bit|low-bit|nf4|compress|prun",
    "co_can_thiep": r"distributed alignment|\bdas\b|interchange|causal abstraction|activation patching|patching|steering|sparse autoencoder|\bsae\b|prob(e|ing)|circuit|mechanis",
    "co_causalgym": r"causalgym",
    "co_pythia_checkpoint": r"pythia|checkpoint",
}
gop = {}
for line in open(F_CAND):
    r = json.loads(line); k = chuan(r["title"])
    if not k: continue
    g = gop.setdefault(k, {"title": r["title"], "year": r["year"], "venue": r["venue"], "abstract": None, "arxiv": None, "doi": None,
                           "url": r["url"], "authors": r["authors"], "nguon": [], "goc": []})
    for f in ("abstract", "arxiv", "doi", "year", "venue"):
        if not g[f] and r.get(f): g[f] = r[f]
    if r["nguon"] not in g["nguon"]: g["nguon"].append(r["nguon"])
    if r["goc"] not in g["goc"]: g["goc"].append(r["goc"])
F_DEDUP = os.path.join(OUT_DIR, "candidates_dedup.jsonl")
with open(F_DEDUP, "w") as f:
    for g in gop.values():
        text = f"{g['title']} {g['abstract'] or ''}".lower()
        for name, rx in RX.items(): g[name] = bool(re.search(rx, text))
        g["uu_tien"] = int(g["co_luong_tu_hoa"]) * 2 + int(g["co_can_thiep"]) + int(g["co_causalgym"]) * 2 + int(g["co_pythia_checkpoint"])
        f.write(json.dumps(g, ensure_ascii=False) + "\n")
ds = sorted(gop.values(), key=lambda g: -g["uu_tien"])
print("bài sau gộp:", len(ds), "| có cả lượng tử hoá và can thiệp:", sum(g["co_luong_tu_hoa"] and g["co_can_thiep"] for g in ds),
      "| thiếu tóm tắt:", sum(not g["abstract"] for g in ds))
for g in ds[:15]:
    print(g["uu_tien"], g["year"], "|", (g["title"] or "")[:100], "|", ",".join(g["goc"])[:60])

## Ô 8. Đóng gói

Tạo `OUT_DIR.zip` (gồm `seeds.jsonl`, `candidates.jsonl`, `candidates_dedup.jsonl`, `errors.jsonl`, `raw/`). Mang file zip vào phiên chat.

In [ ]:
n_err = sum(1 for _ in open(F_ERR))
tom_tat = {"ngay_chay_utc": time.strftime("%Y-%m-%d %H:%M", time.gmtime()), "toy": TOY_TEST, "so_bai_sau_gop": len(gop), "so_loi": n_err,
           "seeds": [json.loads(l) for l in open(F_SEED)]}
json.dump(tom_tat, open(os.path.join(OUT_DIR, "tom_tat.json"), "w"), ensure_ascii=False, indent=1)
zip_path = shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
print("số lỗi:", n_err, "| file mang về:", zip_path, round(os.path.getsize(zip_path) / 1e6, 2), "MB")
if n_err:
    print("Có lỗi: không sao, cứ mang zip về; errors.jsonl cho biết mã nào không tra được.")